# Upper-CEFR Diagnostic Analysis

This notebook analyzes the linguistic characteristics of the **Balanced CEFR Steering Subset** used in the thesis, with particular emphasis on the separability of the upper CEFR levels **B2, C1, and C2**.

The analysis investigates whether the limited performance observed at the highest CEFR levels can be related to distributional overlap in the training data.

The notebook computes lexical, syntactic, readability, text-length, topic-distribution, and duplicate-text diagnostics across all six CEFR levels.

Particular attention is given to:

- B2 / C1 / C2 feature distributions;
- standardized pairwise differences between upper CEFR levels;
- text-length distributions;
- topic coverage;
- exact duplicate rates;
- linguistic features that provide the strongest upper-level separation.

The analysis uses the **Balanced CEFR Steering Subset**, containing 5,568 texts with 928 examples per CEFR level.

In [ ]:
# ============================================================
# 0. INSTALL / IMPORT
# ============================================================
!pip -q install spacy textstat
!python -m spacy download en_core_web_sm -q

import os
import re
import gc
import math
import numpy as np
import pandas as pd
from collections import Counter
import spacy
import textstat
from google.colab import drive
from IPython.display import display

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 14.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 38.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
# ============================================================
# 1. Mount Google Drive
# ============================================================
drive.mount('/content/drive')

# Explicit Paths
INPUT_CSV_PATH = "/content/drive/MyDrive/Your_Path/"
    "balanced_cefr_steering_subset.csv"
OUTPUT_CSV_PATH = "/content/drive/MyDrive/Your_Path/"
    "upper_cefr_diagnostic_features.csv"


Mounted at /content/drive


In [ ]:
# ============================================================
# 2. CONFIGURATION
# ============================================================
CEFR_ORDER = ["A1", "A2", "B1", "B2", "C1", "C2"]

# ============================================================
# 3. LOAD DATASET
# ============================================================
print("Loading dataset...")

df = pd.read_csv(INPUT_CSV_PATH)

required_columns = [
    "text_id",
    "level",
    "cefr",
    "clean_text",
    "topic_title"
]

missing = [c for c in required_columns if c not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df.dropna(subset=["cefr", "clean_text", "topic_title"]).copy()

df["cefr"] = df["cefr"].astype(str).str.upper().str.strip()
df["clean_text"] = df["clean_text"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
df["topic_title"] = df["topic_title"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()

# Keep only valid CEFR labels
df = df[df["cefr"].isin(CEFR_ORDER)].copy()

print(f"Dataset size: {len(df):,} rows")

print("\nCEFR distribution:")
display(
    df["cefr"]
    .value_counts()
    .reindex(CEFR_ORDER)
    .rename("count")
    .to_frame()
)


Loading dataset...
Dataset size: 5,568 rows

CEFR distribution:


,count
cefr,
A1,928
A2,928
B1,928
B2,928
C1,928
C2,928


In [ ]:
# ============================================================
# 4. LOAD NLP MODEL
# ============================================================
print("\nLoading spaCy...")
nlp = spacy.load("en_core_web_sm")
nlp.max_length = max(nlp.max_length, int(df["clean_text"].str.len().max()) + 100)

# ============================================================
# 5. BASIC TEXT ANALYSIS FUNCTIONS
# ============================================================
WORD_RE = re.compile(r"\b[A-Za-z]+(?:'[A-Za-z]+)?\b")

def get_words(text):
    return WORD_RE.findall(text.lower())

def lexical_metrics(text):
    words = get_words(text)
    n = len(words)

    if n == 0:
        return {
            "word_count": 0,
            "unique_word_count": 0,
            "ttr": 0.0,
            "root_ttr": 0.0,
            "avg_word_length": 0.0,
            "long_word_ratio_7": 0.0,
            "long_word_ratio_9": 0.0
        }

    unique_words = set(words)
    word_lengths = np.array([len(w) for w in words])

    return {
        "word_count": n,
        "unique_word_count": len(unique_words),
        "ttr": len(unique_words) / n,
        "root_ttr": len(unique_words) / np.sqrt(n),
        "avg_word_length": word_lengths.mean(),
        "long_word_ratio_7": np.mean(word_lengths >= 7),
        "long_word_ratio_9": np.mean(word_lengths >= 9)
    }

def sentence_metrics(doc):
    sentences = [sent for sent in doc.sents if sent.text.strip()]

    if len(sentences) == 0:
        return {
            "sentence_count": 0,
            "avg_sentence_words": 0.0,
            "median_sentence_words": 0.0,
            "max_sentence_words": 0,
            "sentence_length_std": 0.0
        }

    sentence_lengths = []
    for sent in sentences:
        words = [token for token in sent if token.is_alpha]
        sentence_lengths.append(len(words))

    arr = np.array(sentence_lengths)
    return {
        "sentence_count": len(sentences),
        "avg_sentence_words": arr.mean(),
        "median_sentence_words": np.median(arr),
        "max_sentence_words": arr.max(),
        "sentence_length_std": arr.std()
    }

# ============================================================
# 6. SYNTACTIC COMPLEXITY
# ============================================================
def syntactic_metrics(doc):
    non_punct = [token for token in doc if not token.is_punct]

    if not non_punct:
        return {
            "avg_dependency_distance": 0.0,
            "median_dependency_distance": 0.0,
            "max_dependency_distance": 0,
            "avg_tree_depth": 0.0,
            "max_tree_depth": 0,
            "subordinate_clause_ratio": 0.0,
            "subordinate_clause_count": 0,
            "conjunction_ratio": 0.0,
            "relative_clause_count": 0,
            "passive_count": 0
        }

    # Dependency distance
    distances = []
    for token in non_punct:
        if token.head.i == token.i:
            continue
        distances.append(abs(token.i - token.head.i))

    avg_dep = np.mean(distances) if distances else 0
    median_dep = np.median(distances) if distances else 0
    max_dep = np.max(distances) if distances else 0

    # Dependency tree depth
    depths = []
    for token in non_punct:
        depth = 0
        current = token
        visited = set()
        while current.head.i != current.i and current.i not in visited:
            visited.add(current.i)
            depth += 1
            current = current.head
        depths.append(depth)

    avg_depth = np.mean(depths) if depths else 0
    max_depth = np.max(depths) if depths else 0

    # Clause / syntactic markers
    subordinate_deps = {"advcl", "ccomp", "xcomp", "acl", "relcl"}
    subordinate_tokens = [t for t in doc if t.dep_ in subordinate_deps]
    relative_clauses = [t for t in doc if t.dep_ == "relcl"]
    passive_tokens = [t for t in doc if t.dep_ in {"auxpass", "nsubjpass"}]
    conjunctions = [t for t in doc if t.dep_ in {"conj", "cc"}]

    total_tokens = len(non_punct)

    return {
        "avg_dependency_distance": avg_dep,
        "median_dependency_distance": median_dep,
        "max_dependency_distance": max_dep,
        "avg_tree_depth": avg_depth,
        "max_tree_depth": max_depth,
        "subordinate_clause_ratio": len(subordinate_tokens) / total_tokens if total_tokens else 0,
        "subordinate_clause_count": len(subordinate_tokens),
        "conjunction_ratio": len(conjunctions) / total_tokens if total_tokens else 0,
        "relative_clause_count": len(relative_clauses),
        "passive_count": len(passive_tokens)
    }

# ============================================================
# 7. POS DISTRIBUTION
# ============================================================
def pos_metrics(doc):
    tokens = [token for token in doc if token.is_alpha]
    if not tokens:
        return {
            "noun_ratio": 0.0, "verb_ratio": 0.0, "adj_ratio": 0.0,
            "adv_ratio": 0.0, "pronoun_ratio": 0.0, "proper_noun_ratio": 0.0
        }
    total = len(tokens)
    counts = Counter(token.pos_ for token in tokens)
    return {
        "noun_ratio": counts["NOUN"] / total,
        "verb_ratio": counts["VERB"] / total,
        "adj_ratio": counts["ADJ"] / total,
        "adv_ratio": counts["ADV"] / total,
        "pronoun_ratio": counts["PRON"] / total,
        "proper_noun_ratio": counts["PROPN"] / total
    }

# ============================================================
# 8. READABILITY
# ============================================================
def readability_metrics(text):
    try: flesch = textstat.flesch_reading_ease(text)
    except: flesch = np.nan

    try: grade = textstat.flesch_kincaid_grade(text)
    except: grade = np.nan

    try: ari = textstat.automated_readability_index(text)
    except: ari = np.nan

    try: dale = textstat.dale_chall_readability_score(text)
    except: dale = np.nan

    return {
        "flesch_reading_ease": flesch,
        "flesch_kincaid_grade": grade,
        "automated_readability_index": ari,
        "dale_chall": dale
    }


Loading spaCy...


In [ ]:
# ============================================================
# 9. RUN ROW-LEVEL ANALYSIS
# ============================================================
print("\nRunning linguistic analysis...")
print("This may take several minutes depending on dataset size.")

analysis_records = []

for idx, row in enumerate(df.itertuples(index=False), start=1):
    text = row.clean_text
    topic = row.topic_title
    doc = nlp(text)

    record = {
        "text_id": row.text_id,
        "level": row.level,
        "cefr": row.cefr,
        "topic_title": topic,
        "text_char_count": len(text),
    }

    record.update(lexical_metrics(text))
    record.update(sentence_metrics(doc))
    record.update(syntactic_metrics(doc))
    record.update(pos_metrics(doc))
    record.update(readability_metrics(text))

    analysis_records.append(record)

    if idx % 1000 == 0:
        print(f"Processed {idx:,} / {len(df):,}")

analysis_df = pd.DataFrame(analysis_records)

# ============================================================
# 10. TOPIC-LEVEL INFORMATION
# ============================================================
print("\nComputing topic-level statistics...")

topic_counts = (
    analysis_df.groupby("topic_title")
    .agg(topic_sample_count=("text_id", "count"), topic_cefr_count=("cefr", "nunique"))
    .reset_index()
)

analysis_df = analysis_df.merge(topic_counts, on="topic_title", how="left")

topic_cefr_counts = analysis_df.groupby(["topic_title", "cefr"]).size().reset_index(name="topic_cefr_count")
topic_cefr_counts["topic_cefr_proportion"] = topic_cefr_counts.groupby("topic_title")["topic_cefr_count"].transform(lambda x: x / x.sum())

max_topic_prop = topic_cefr_counts.groupby("topic_title")["topic_cefr_proportion"].max().rename("topic_cefr_max_proportion").reset_index()
analysis_df = analysis_df.merge(max_topic_prop, on="topic_title", how="left")

# ============================================================
# 11. DUPLICATE / NEAR-DUPLICATE INDICATORS
# ============================================================
duplicate_map = dict(zip(df["text_id"], df.groupby("clean_text")["text_id"].transform("count")))
analysis_df["text_exact_duplicate_count"] = analysis_df["text_id"].map(duplicate_map)
analysis_df["is_exact_duplicate"] = (analysis_df["text_exact_duplicate_count"] > 1).astype(int)

# ============================================================
# 12. LEVEL ↔ CEFR RELATIONSHIP
# ============================================================
print("\n============================================================")
print("LEVEL ↔ CEFR DISTRIBUTION")
print("============================================================")
level_cefr_table = pd.crosstab(df["level"], df["cefr"]).reindex(columns=CEFR_ORDER, fill_value=0)
display(level_cefr_table)

# ============================================================
# 13. MAIN CEFR SUMMARY
# ============================================================
metric_columns = [
    "text_char_count", "word_count", "unique_word_count",
    "ttr", "root_ttr", "avg_word_length", "long_word_ratio_7", "long_word_ratio_9",
    "sentence_count", "avg_sentence_words", "median_sentence_words", "max_sentence_words", "sentence_length_std",
    "avg_dependency_distance", "median_dependency_distance", "max_dependency_distance", "avg_tree_depth", "max_tree_depth",
    "subordinate_clause_ratio", "subordinate_clause_count", "conjunction_ratio", "relative_clause_count", "passive_count",
    "noun_ratio", "verb_ratio", "adj_ratio", "adv_ratio", "pronoun_ratio", "proper_noun_ratio",
    "flesch_reading_ease", "flesch_kincaid_grade", "automated_readability_index", "dale_chall"
]

print("\n============================================================")
print("CEFR DESCRIPTIVE STATISTICS")
print("============================================================")
cefr_summary = analysis_df.groupby("cefr")[metric_columns].agg(["mean", "median", "std"]).reindex(CEFR_ORDER)
display(cefr_summary.round(3))

# ============================================================
# 14. SIMPLER MEAN TABLE
# ============================================================
print("\n============================================================")
print("CEFR MEAN VALUES — MAIN DIAGNOSTIC TABLE")
print("============================================================")
cefr_means = analysis_df.groupby("cefr")[metric_columns].mean().reindex(CEFR_ORDER)
display(cefr_means.round(3))

# ============================================================
# 15. C1/C2 VS B2 COMPARISON
# ============================================================
print("\n============================================================")
print("B2 vs C1 vs C2")
print("============================================================")
focus_metrics = [
    "word_count", "ttr", "root_ttr", "avg_word_length", "long_word_ratio_7", "long_word_ratio_9",
    "avg_sentence_words", "median_sentence_words", "max_sentence_words",
    "avg_dependency_distance", "avg_tree_depth", "subordinate_clause_ratio", "relative_clause_count", "passive_count",
    "flesch_reading_ease", "flesch_kincaid_grade"
]

focus_metrics = [x for x in focus_metrics if x in analysis_df.columns]
b2_c1_c2 = analysis_df[analysis_df["cefr"].isin(["B2", "C1", "C2"])].groupby("cefr")[focus_metrics].agg(["mean", "median", "std"]).reindex(["B2", "C1", "C2"])
display(b2_c1_c2.round(3))

# ============================================================
# 16. C1/C2 OVERLAP ANALYSIS
# ============================================================
print("\n============================================================")
print("C1/C2 DISTRIBUTIONAL OVERLAP")
print("============================================================")
overlap_metrics = [
    "word_count", "ttr", "root_ttr", "avg_word_length", "avg_sentence_words",
    "avg_dependency_distance", "avg_tree_depth", "subordinate_clause_ratio",
    "flesch_reading_ease", "flesch_kincaid_grade"
]

overlap_rows = []
for metric in overlap_metrics:
    if metric not in analysis_df.columns:
        continue

    groups = {cefr: analysis_df.loc[analysis_df["cefr"] == cefr, metric].dropna() for cefr in CEFR_ORDER}
    row = {"metric": metric}

    c1, c2, b2 = groups["C1"], groups["C2"], groups["B2"]
    row["B2_mean"], row["C1_mean"], row["C2_mean"] = b2.mean(), c1.mean(), c2.mean()

    if len(b2) > 0:
        row["C1_inside_B2_range_%"] = ((c1 >= b2.min()) & (c1 <= b2.max())).mean() * 100
        row["C2_inside_B2_range_%"] = ((c2 >= b2.min()) & (c2 <= b2.max())).mean() * 100

    overlap_rows.append(row)

overlap_table = pd.DataFrame(overlap_rows)
display(overlap_table.round(3))

# ============================================================
# 17. TEXT LENGTH BIN ANALYSIS
# ============================================================
print("\n============================================================")
print("TEXT LENGTH DISTRIBUTION")
print("============================================================")
length_bins = [0, 50, 100, 150, 200, 250, 300, 400, 500, 750, 1000, np.inf]
length_labels = ["<50", "50-99", "100-149", "150-199", "200-249", "250-299", "300-399", "400-499", "500-749", "750-999", "1000+"]

analysis_df["word_length_bin"] = pd.cut(analysis_df["word_count"], bins=length_bins, labels=length_labels, right=False)
length_cefr_table = pd.crosstab(analysis_df["cefr"], analysis_df["word_length_bin"], normalize="index") * 100
display(length_cefr_table.round(2))

# ============================================================
# 18. TOPIC DISTRIBUTION
# ============================================================
print("\n============================================================")
print("TOPIC DISTRIBUTION")
print("============================================================")
topic_summary = analysis_df.groupby("cefr").agg(
    unique_topics=("topic_title", "nunique"),
    mean_samples_per_topic=("topic_sample_count", "mean"),
    median_samples_per_topic=("topic_sample_count", "median"),
    mean_topic_cefr_max_proportion=("topic_cefr_max_proportion", "mean")
).reindex(CEFR_ORDER)
display(topic_summary.round(3))

# ============================================================
# 19. TOPIC BALANCE
# ============================================================
topic_cefr_distribution = pd.crosstab(analysis_df["topic_title"], analysis_df["cefr"]).reindex(columns=CEFR_ORDER, fill_value=0)
print("\nNumber of topics represented at each CEFR level:")
topic_presence = (topic_cefr_distribution > 0).sum(axis=0)
display(topic_presence.rename("number_of_topics").to_frame())

# ============================================================
# 20. EXACT DUPLICATE ANALYSIS
# ============================================================
print("\n============================================================")
print("DUPLICATE TEXT ANALYSIS")
print("============================================================")
duplicate_summary = analysis_df.groupby("cefr").agg(
    samples=("text_id", "count"),
    duplicated_samples=("is_exact_duplicate", "sum")
).reindex(CEFR_ORDER)
duplicate_summary["duplicate_rate_%"] = (duplicate_summary["duplicated_samples"] / duplicate_summary["samples"] * 100)
display(duplicate_summary.round(3))

# ============================================================
# 21. CEFR SEPARABILITY SCORE
# ============================================================
print("\n============================================================")
print("CEFR SEPARABILITY")
print("============================================================")
separation_metrics = [
    "word_count", "ttr", "root_ttr", "avg_word_length", "long_word_ratio_7",
    "avg_sentence_words", "avg_dependency_distance", "avg_tree_depth",
    "subordinate_clause_ratio", "relative_clause_count", "passive_count",
    "flesch_reading_ease", "flesch_kincaid_grade"
]

separation_rows = []
for metric in separation_metrics:
    if metric not in analysis_df.columns: continue

    def standardized_difference(a, b):
        a, b = np.asarray(a.dropna()), np.asarray(b.dropna())
        if len(a) < 2 or len(b) < 2: return np.nan
        var_a, var_b = np.var(a, ddof=1), np.var(b, ddof=1)
        pooled_sd = np.sqrt(((len(a)-1) * var_a + (len(b)-1) * var_b) / (len(a)+len(b)-2))
        return 0 if pooled_sd == 0 else abs(a.mean() - b.mean()) / pooled_sd

    b2, c1, c2 = analysis_df.loc[analysis_df["cefr"] == "B2", metric].dropna(), analysis_df.loc[analysis_df["cefr"] == "C1", metric].dropna(), analysis_df.loc[analysis_df["cefr"] == "C2", metric].dropna()
    separation_rows.append({
        "metric": metric,
        "B2_vs_C1_effect": standardized_difference(b2, c1),
        "C1_vs_C2_effect": standardized_difference(c1, c2),
        "B2_vs_C2_effect": standardized_difference(b2, c2)
    })

separation_table = pd.DataFrame(separation_rows).sort_values("C1_vs_C2_effect", ascending=False)
display(separation_table.round(3))

# ============================================================
# 22. WHICH FEATURES DIFFER MOST BETWEEN B2/C1/C2?
# ============================================================
print("\n============================================================")
print("C1/C2 DIAGNOSTIC RANKING")
print("============================================================")
diagnostic_ranking = separation_table.copy()
diagnostic_ranking["average_upper_level_separation"] = diagnostic_ranking[["B2_vs_C1_effect", "C1_vs_C2_effect", "B2_vs_C2_effect"]].mean(axis=1)
diagnostic_ranking = diagnostic_ranking.sort_values("average_upper_level_separation", ascending=False)
display(diagnostic_ranking.round(3))

# ============================================================
# 23. SAVE ANALYSIS DATASET
# ============================================================
analysis_output = analysis_df.merge(df[["text_id", "clean_text"]], on="text_id", how="left")
first_columns = ["text_id", "level", "cefr", "topic_title", "clean_text"]
remaining_columns = [c for c in analysis_output.columns if c not in first_columns]
analysis_output = analysis_output[first_columns + remaining_columns]

analysis_output.to_csv(OUTPUT_CSV_PATH, index=False)

print("\n============================================================")
print("ANALYSIS COMPLETE")
print("============================================================")
print(f"Input : {INPUT_CSV_PATH}")
print(f"Output: {OUTPUT_CSV_PATH}")
print(f"Rows  : {len(analysis_output):,}")
print(f"Columns: {len(analysis_output.columns):,}")

# ============================================================
# 24. FINAL EXECUTIVE TABLE
# ============================================================
print("\n============================================================")
print("FINAL C1/C2 DIAGNOSTIC TABLE")
print("============================================================")
final_columns = [
    "word_count", "ttr", "avg_word_length", "avg_sentence_words",
    "avg_dependency_distance", "avg_tree_depth", "subordinate_clause_ratio",
    "relative_clause_count", "passive_count", "flesch_reading_ease", "flesch_kincaid_grade"
]

final_table = analysis_output.groupby("cefr")[final_columns].agg(["mean", "median"]).reindex(CEFR_ORDER)
display(final_table.round(3))

print("\n============================================================")
print("B2 / C1 / C2 SEPARABILITY TABLE")
print("============================================================")
display(diagnostic_ranking.head(15).round(3))


Running linguistic analysis...
This may take several minutes depending on dataset size.
Processed 1,000 / 5,568
Processed 2,000 / 5,568
Processed 3,000 / 5,568
Processed 4,000 / 5,568
Processed 5,000 / 5,568

Computing topic-level statistics...

LEVEL ↔ CEFR DISTRIBUTION


cefr,A1,A2,B1,B2,C1,C2
level,,,,,,
1,487,0,0,0,0,0
2,259,0,0,0,0,0
3,182,0,0,0,0,0
4,0,483,0,0,0,0
5,0,272,0,0,0,0
6,0,173,0,0,0,0
7,0,0,521,0,0,0
8,0,0,241,0,0,0
9,0,0,166,0,0,0



CEFR DESCRIPTIVE STATISTICS


text_char_count                 word_count                 \
                mean median      std       mean median     std   
cefr                                                             
A1           218.265  192.5  110.776     40.567   36.0  20.030   
A2           356.694  347.0  128.026     66.347   65.0  23.482   
B1           510.606  511.0  135.819     93.931   95.0  25.589   
B2           730.393  719.0  215.233    131.508  130.0  38.044   
C1           936.748  943.0  245.212    164.322  167.0  41.969   
C2           964.230  964.5  267.114    170.786  171.0  45.750   

     unique_word_count                   ttr  ... flesch_reading_ease  \
                  mean median     std   mean  ...                 std   
cefr                                          ...                       
A1              29.332   28.0  11.927  0.749  ...              13.276   
A2              45.858   45.0  13.412  0.708  ...              13.109   
B1              63.749   64.0  14.467  0.690  ...              15.159   
B2              83.046   84.0  21.392  0.642  ...              13.191   
C1             100.091  102.0  22.050  0.620  ...              13.320   
C2             105.025  106.0  24.813  0.625  ...              16.713   

     flesch_kincaid_grade               automated_readability_index         \
                     mean median    std                        mean median   
cefr                                                                         
A1                  4.228  3.826  2.819                       3.807  3.310   
A2                  5.144  4.792  2.713                       4.501  3.931   
B1                  7.031  6.601  3.395                       6.631  5.959   
B2                  8.226  7.701  3.360                       8.051  7.458   
C1                  9.008  8.563  2.961                       9.277  8.750   
C2                  9.693  9.176  5.072                      10.226  9.528   

            dale_chall                
        std       mean median    std  
cefr                                  
A1    3.732      7.766  7.482  1.815  
A2    3.524      7.780  7.520  1.525  
B1    4.184      8.087  7.965  1.487  
B2    4.135      8.240  8.093  1.319  
C1    3.556      8.537  8.452  1.292  
C2    6.438      8.952  8.833  1.466  

[6 rows x 99 columns]


CEFR MEAN VALUES — MAIN DIAGNOSTIC TABLE


...

,metric,B2_vs_C1_effect,C1_vs_C2_effect,B2_vs_C2_effect,average_upper_level_separation
0,word_count,0.819,0.147,0.934,0.633
2,root_ttr,0.622,0.224,0.804,0.550
10,passive_count,0.483,0.227,0.689,0.467
7,avg_tree_depth,0.354,0.315,0.607,0.425
9,relative_clause_count,0.370,0.225,0.580,0.391
3,avg_word_length,0.407,0.180,0.233,0.273
5,avg_sentence_words,0.186,0.260,0.369,0.272
12,flesch_kincaid_grade,0.247,0.165,0.341,0.251
6,avg_dependency_distance,0.172,0.213,0.354,0.246
1,ttr,0.327,0.085,0.241,0.218



ANALYSIS COMPLETE
Input : /content/drive/MyDrive/Mohammd_Thesis/subsets/efcamdat_balanced_subset_training.csv
Output: /content/drive/MyDrive/Mohammd_Thesis/subsets/efcamdat_balanced_subset_analysis.csv
Rows  : 5,568
Columns: 44

FINAL C1/C2 DIAGNOSTIC TABLE


word_count           ttr        avg_word_length         \
           mean median   mean median            mean median   
cefr                                                          
A1       40.567   36.0  0.749  0.750           4.124  4.092   
A2       66.347   65.0  0.708  0.707           4.149  4.132   
B1       93.931   95.0  0.690  0.685           4.284  4.252   
B2      131.508  130.0  0.642  0.640           4.403  4.376   
C1      164.322  167.0  0.620  0.616           4.554  4.532   
C2      170.786  171.0  0.625  0.620           4.489  4.494   

     avg_sentence_words         avg_dependency_distance         ...  \
                   mean  median                    mean median  ...   
cefr                                                            ...   
A1                8.854   7.714                   1.949  1.868  ...   
A2               10.251   9.118                   2.076  2.022  ...   
B1               13.665  12.429                   2.236  2.193  ...   
B2               15.719  14.388                   2.292  2.248  ...   
C1               16.999  15.727                   2.349  2.313  ...   
C2               19.523  17.696                   2.432  2.361  ...   

     subordinate_clause_ratio        relative_clause_count         \
                         mean median                  mean median   
cefr                                                                
A1                      0.029  0.023                 0.106    0.0   
A2                      0.050  0.046                 0.350    0.0   
B1                      0.066  0.067                 0.759    0.0   
B2                      0.074  0.074                 1.419    1.0   
C1                      0.071  0.071                 1.982    2.0   
C2                      0.073  0.074                 2.373    2.0   

     passive_count        flesch_reading_ease         flesch_kincaid_grade  \
              mean median                mean  median                 mean   
cefr                                                                         
A1           0.137    0.0              81.852  82.952                4.228   
A2           0.647    0.0              78.001  79.383                5.144   
B1           1.081    0.0              70.440  71.966                7.031   
B2           1.404    0.0              65.239  66.406                8.226   
C1           2.676    2.0              61.300  62.823                9.008   
C2           3.447    2.0              60.648  61.724                9.693   

             
     median  
cefr         
A1    3.826  
A2    4.792  
B1    6.601  
B2    7.701  
C1    8.563  
C2    9.176  

[6 rows x 22 columns]


B2 / C1 / C2 SEPARABILITY TABLE


,metric,B2_vs_C1_effect,C1_vs_C2_effect,B2_vs_C2_effect,average_upper_level_separation
0,word_count,0.819,0.147,0.934,0.633
2,root_ttr,0.622,0.224,0.804,0.550
10,passive_count,0.483,0.227,0.689,0.467
7,avg_tree_depth,0.354,0.315,0.607,0.425
9,relative_clause_count,0.370,0.225,0.580,0.391
3,avg_word_length,0.407,0.180,0.233,0.273
5,avg_sentence_words,0.186,0.260,0.369,0.272
12,flesch_kincaid_grade,0.247,0.165,0.341,0.251
6,avg_dependency_distance,0.172,0.213,0.354,0.246
1,ttr,0.327,0.085,0.241,0.218


## Upper-Level Diagnostic Summary

The analysis shows substantially weaker separation between **C1 and C2** than
between **B2 and C1** for most examined linguistic features.

Examples of absolute standardized C1–C2 differences include:

| Feature | C1 vs C2 |
|---|---:|
| Average tree depth | 0.315 |
| Average sentence length | 0.260 |
| Passive count | 0.227 |
| Relative-clause count | 0.225 |
| Root TTR | 0.224 |
| Average dependency distance | 0.213 |
| Average word length | 0.180 |
| Word count | 0.147 |
| TTR | 0.085 |
| Flesch Reading Ease | 0.043 |
| Subordinate-clause ratio | 0.063 |

These results indicate that the common lexical, syntactic, and readability
features examined here provide only limited separation between C1 and C2 in the
balanced steering subset.

This offers a data-level explanation for why controlling and distinguishing
the upper CEFR tiers is more difficult than separating lower proficiency levels.